# 10 · Métricas de calidad sobre la prueba final

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Calcula las métricas de calidad oficiales sobre la prueba de 10.000 comentarios del notebook 06: accuracy, precisión, recall y F1 (macro y ponderado), F1 por emoción, intervalos de confianza del 95 % (bootstrap), % de respuestas inválidas, ROUGE-L y BLEU-1. Reutiliza las respuestas ya generadas en el notebook 09 y genera las faltantes con el mismo prompt y parámetros. Esta ejecución corresponde a Gemma-2 2B.

*Nombre durante el desarrollo: `07b_Evaluacion_Prueba_Final` (algunas salidas conservan esa numeración).*

## 1. Dependencias (las mismas del notebook 09)

In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes "huggingface_hub<2.0" scikit-learn "pandas<3" pyarrow rouge-score nltk
!pip uninstall -y -q torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 920.1 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency co

## 2. Configuración

In [2]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

TU_USUARIO_HF = "santiagortegasegura"

MODELOS_DISPONIBLES = {
    "qwen_1_5b":  "Qwen/Qwen2.5-1.5B-Instruct",
    "llama_3b":   "meta-llama/Llama-3.2-3B-Instruct",
    "qwen_3b":    "Qwen/Qwen2.5-3B-Instruct",
    "phi35_mini": "microsoft/Phi-3.5-mini-instruct",
    "phi3_mini":  "microsoft/Phi-3-mini-4k-instruct",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3",
    "qwen_7b":    "Qwen/Qwen2.5-7B-Instruct",
    "llama_8b":   "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "gemma2_2b":  "google/gemma-2-2b-it",
    "gemma2_9b":  "google/gemma-2-9b-it",
}

CLAVE_MODELO = "gemma2_2b"
MODEL_ID = MODELOS_DISPONIBLES[CLAVE_MODELO]

REPO_DATOS = f"{TU_USUARIO_HF}/emotion-futbol-data"
REPO_CHECKPOINTS = f"{TU_USUARIO_HF}/emotion-futbol-checkpoints"

COLUMNA_TEXTO = "text"
MAX_LENGTH = 112
EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
TAMANO_LOTE = 16

print(f"Evaluando: {CLAVE_MODELO} -> {MODEL_ID}")

Evaluando: gemma2_2b -> google/gemma-2-2b-it


## 3. Cargar la prueba final y las predicciones ya calculadas en el notebook 09

In [3]:
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

def bajar(nombre):
    return pd.read_csv(hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename=nombre),
                       keep_default_na=False)

df_test = bajar("final/prueba_final_10k.csv")
df_mapa = bajar("final/auxiliar/mapa_prueba_final.csv")
df_prueba_04b = bajar("04b_validacion_final_combinada.csv").reset_index(drop=True)
df_pred_07 = bajar(f"predicciones/07_predicciones_{CLAVE_MODELO}.csv")
df_res_07 = bajar("07_resultados_finales.csv")

assert (df_mapa['comment_id'].values == df_test['comment_id'].values).all(), "El mapa no corresponde a la prueba final."
df_test['indice_04b'] = df_mapa['indice_04b'].astype(int).values
print(f"Prueba final: {len(df_test)} | Predicciones del 07: {len(df_pred_07)}")

assert len(df_pred_07) == len(df_prueba_04b)
coinciden = df_pred_07[COLUMNA_TEXTO].astype(str).values == df_prueba_04b[COLUMNA_TEXTO].astype(str).values
assert coinciden.all(), "El orden de las predicciones del 07 no coincide -- no se pueden reutilizar."

fila_07 = df_res_07[df_res_07['modelo'] == CLAVE_MODELO]
assert len(fila_07) == 1, f"No se encontró '{CLAVE_MODELO}' en 07_resultados_finales.csv."
fila_07 = fila_07.iloc[0]

prueba_final_10k.csv:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

mapa_prueba_final.csv:   0%|          | 0.00/318k [00:00<?, ?B/s]

04b_validacion_final_combinada.csv:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

07_predicciones_gemma2_2b.csv:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

07_resultados_finales.csv:   0%|          | 0.00/5.15k [00:00<?, ?B/s]

Prueba final: 10000 | Predicciones del 07: 10000


## 4. Reutilizar las predicciones existentes

In [4]:
df_test['prediccion'] = None
df_test['prediccion_cruda'] = None

tiene_prediccion = df_test['indice_04b'] >= 0
indices = df_test.loc[tiene_prediccion, 'indice_04b'].values

df_test.loc[tiene_prediccion, 'prediccion'] = df_pred_07.loc[indices, 'prediccion'].values
df_test.loc[tiene_prediccion, 'prediccion_cruda'] = df_pred_07.loc[indices, 'prediccion_cruda'].values
assert (df_test.loc[tiene_prediccion, 'label'].values == df_pred_07.loc[indices, 'label'].values).all()

print(f"Predicciones reutilizadas: {tiene_prediccion.sum()} | por generar: {(~tiene_prediccion).sum()}")

Predicciones reutilizadas: 9311 | por generar: 689


## 5. Cargar el modelo base + el adaptador LoRA (igual que en el notebook 09)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import snapshot_download

ruta_relativa_adaptador = fila_07['checkpoint']
print(f"Adaptador: {ruta_relativa_adaptador}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

modelo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", attn_implementation="eager",
)
ruta_adaptador = snapshot_download(
    repo_id=REPO_CHECKPOINTS, repo_type="dataset", allow_patterns=[f"{ruta_relativa_adaptador}/*"],
)
modelo = PeftModel.from_pretrained(modelo_base, f"{ruta_adaptador}/{ruta_relativa_adaptador}")
modelo.eval()
print(f"Modelo cargado: {CLAVE_MODELO}")

Adaptador: gemma2_2b/adapter_final


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Modelo cargado: gemma2_2b


## 6. Prompt y parseo (los mismos del notebook 09)

In [6]:
def formatear_prompt_inferencia(texto):
    return (
        f"Clasifica la emoción del siguiente comentario de fútbol en una de estas categorías: "
        f"{', '.join(EMOCIONES)}.\n\nComentario: {texto}\n\nEmoción:"
    )

def parsear_emocion(texto_generado):
    limpio = texto_generado.strip().lower().split()[0] if texto_generado.strip() else ""
    limpio = limpio.strip(".,;:!¡¿?\"'")
    return limpio if limpio in EMOCIONES else "invalido"

print(formatear_prompt_inferencia("Qué golazo, no lo puedo creer"))

Clasifica la emoción del siguiente comentario de fútbol en una de estas categorías: alegria, tristeza, enojo, miedo, sorpresa, burla, neutral.

Comentario: Qué golazo, no lo puedo creer

Emoción:


## 7. Generar las predicciones que faltan

In [7]:
from tqdm.auto import tqdm

pendientes = df_test.index[~tiene_prediccion].tolist()
prompts = [formatear_prompt_inferencia(df_test.at[i, COLUMNA_TEXTO]) for i in pendientes]

for inicio in tqdm(range(0, len(pendientes), TAMANO_LOTE), desc=f"Inferencia {CLAVE_MODELO}"):
    lote_idx = pendientes[inicio:inicio + TAMANO_LOTE]
    lote_prompts = prompts[inicio:inicio + TAMANO_LOTE]
    entradas = tokenizer(lote_prompts, return_tensors="pt", padding=True, truncation=True,
                         max_length=MAX_LENGTH).to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entradas, max_new_tokens=6, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
    for j, idx in enumerate(lote_idx):
        generado = tokenizer.decode(salida[j][entradas['input_ids'].shape[1]:], skip_special_tokens=True)
        df_test.at[idx, 'prediccion_cruda'] = generado
        df_test.at[idx, 'prediccion'] = parsear_emocion(generado)

assert df_test['prediccion'].notna().all(), "Quedaron filas sin predicción."
df_test['prediccion_cruda'] = df_test['prediccion_cruda'].fillna("").astype(str)
n_invalidas = int((df_test['prediccion'] == 'invalido').sum())
print(f"Predicciones inválidas: {n_invalidas} ({100 * n_invalidas / len(df_test):.2f}%)")

Inferencia gemma2_2b:   0%|          | 0/44 [00:00<?, ?it/s]

Predicciones inválidas: 169 (1.69%)


## 8. Métricas de calidad sobre las 10.000

In [8]:
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

y_true = df_test['label']
y_pred = df_test['prediccion']

reporte = classification_report(y_true, y_pred, labels=EMOCIONES, output_dict=True, zero_division=0)
accuracy = accuracy_score(y_true, y_pred)
print(classification_report(y_true, y_pred, labels=EMOCIONES, zero_division=0))

scorer_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
suavizado = SmoothingFunction().method1
rouge_l, bleu1 = [], []
for ref, cand in zip(y_true, df_test['prediccion_cruda']):
    cand, ref = cand.strip().lower(), ref.strip().lower()
    rouge_l.append(scorer_rouge.score(ref, cand)['rougeL'].fmeasure)
    bleu1.append(sentence_bleu([ref.split()], cand.split(), weights=(1.0,), smoothing_function=suavizado)
                 if cand else 0.0)
df_test['rouge_l'] = rouge_l
df_test['bleu1'] = bleu1

N_REMUESTREOS = 1000
rng = np.random.default_rng(42)
yt, yp = y_true.to_numpy(), y_pred.to_numpy()
n = len(yt)
accs, f1s = np.empty(N_REMUESTREOS), np.empty(N_REMUESTREOS)
for i in range(N_REMUESTREOS):
    idx = rng.integers(0, n, n)
    accs[i] = accuracy_score(yt[idx], yp[idx])
    f1s[i] = f1_score(yt[idx], yp[idx], labels=EMOCIONES, average='macro', zero_division=0)
accuracy_ic95 = (float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5)))
f1_macro_ic95 = (float(np.percentile(f1s, 2.5)), float(np.percentile(f1s, 97.5)))

print(f"Accuracy: {accuracy:.4f}  IC95 [{accuracy_ic95[0]:.4f}, {accuracy_ic95[1]:.4f}]")
print(f"F1 macro: {reporte['macro avg']['f1-score']:.4f}  IC95 [{f1_macro_ic95[0]:.4f}, {f1_macro_ic95[1]:.4f}]")
print(f"ROUGE-L: {np.mean(rouge_l):.4f} | BLEU-1: {np.mean(bleu1):.4f}")

matriz = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=EMOCIONES), index=EMOCIONES, columns=EMOCIONES)
matriz.index.name, matriz.columns.name = "real", "predicho"

assert abs(accuracy - reporte['weighted avg']['recall']) < 1e-9, "accuracy != recall_weighted"
assert accuracy_ic95[0] <= accuracy <= accuracy_ic95[1], "La accuracy cae fuera de su IC"
print("Chequeos de consistencia: OK")

              precision    recall  f1-score   support

     alegria       0.88      0.88      0.88      3282
    tristeza       0.85      0.78      0.81       479
       enojo       0.83      0.85      0.84      3104
       miedo       0.70      0.62      0.65        26
    sorpresa       0.73      0.69      0.71       409
       burla       0.87      0.83      0.85      1633
     neutral       0.72      0.67      0.69      1067

   micro avg       0.84      0.82      0.83     10000
   macro avg       0.80      0.76      0.78     10000
weighted avg       0.84      0.82      0.83     10000

Accuracy: 0.8248  IC95 [0.8177, 0.8323]
F1 macro: 0.7762  IC95 [0.7479, 0.7991]
ROUGE-L: 0.3911 | BLEU-1: 0.2303
Chequeos de consistencia: OK


## 9. Guardar resultados en la carpeta `final/` de Hugging Face

In [9]:
from huggingface_hub import upload_file
try:
    from huggingface_hub.errors import EntryNotFoundError
except ImportError:
    from huggingface_hub.utils import EntryNotFoundError

RUTA_RESULTADOS = "07_resultados_finales.csv"

fila = {
    "modelo": CLAVE_MODELO,
    "model_id": MODEL_ID,
    "checkpoint": ruta_relativa_adaptador,
    "n_test": len(df_test),
    "accuracy": accuracy,
    "accuracy_ic95_bajo": accuracy_ic95[0],
    "accuracy_ic95_alto": accuracy_ic95[1],
    "precision_macro": reporte["macro avg"]["precision"],
    "recall_macro": reporte["macro avg"]["recall"],
    "f1_macro": reporte["macro avg"]["f1-score"],
    "f1_macro_ic95_bajo": f1_macro_ic95[0],
    "f1_macro_ic95_alto": f1_macro_ic95[1],
    "precision_weighted": reporte["weighted avg"]["precision"],
    "recall_weighted": reporte["weighted avg"]["recall"],
    "f1_weighted": reporte["weighted avg"]["f1-score"],
    "pct_predicciones_invalidas": 100 * n_invalidas / len(df_test),
    "rouge_l_promedio": float(np.mean(rouge_l)),
    "bleu1_promedio": float(np.mean(bleu1)),
    "latencia_prom_seg": fila_07["latencia_prom_seg"],
    "latencia_lote_prom_seg": fila_07["latencia_lote_prom_seg"],
    "memoria_gpu_pico_mb": fila_07["memoria_gpu_pico_mb"],
    "almacenamiento_adaptador_mb": fila_07["almacenamiento_adaptador_mb"],
}
for emocion in EMOCIONES:
    fila[f"f1_{emocion}"] = reporte.get(emocion, {}).get("f1-score", None)

try:
    df_resultados = bajar(f"final/{RUTA_RESULTADOS}")
    df_resultados = df_resultados[df_resultados['modelo'] != CLAVE_MODELO]
    df_resultados = pd.concat([df_resultados, pd.DataFrame([fila])], ignore_index=True)
except EntryNotFoundError:
    df_resultados = pd.DataFrame([fila])
df_resultados.to_csv(RUTA_RESULTADOS, index=False)

ruta_pred = f"07_predicciones_{CLAVE_MODELO}.csv"
ruta_matriz = f"07_matriz_confusion_{CLAVE_MODELO}.csv"
df_test[[COLUMNA_TEXTO, 'label', 'prediccion', 'prediccion_cruda', 'rouge_l', 'bleu1']].to_csv(ruta_pred, index=False)
matriz.to_csv(ruta_matriz)

for local, remoto in [(RUTA_RESULTADOS, f"final/{RUTA_RESULTADOS}"),
                      (ruta_pred, f"final/predicciones/{ruta_pred}"),
                      (ruta_matriz, f"final/matrices_confusion/{ruta_matriz}")]:
    upload_file(path_or_fileobj=local, path_in_repo=remoto, repo_id=REPO_DATOS, repo_type="dataset")

print(f"Listo. Modelos en final/{RUTA_RESULTADOS}: {sorted(df_resultados['modelo'].tolist())}")
df_resultados[['modelo', 'accuracy', 'f1_macro', 'pct_predicciones_invalidas']]

07_resultados_finales.csv:   0%|          | 0.00/4.23k [00:00<?, ?B/s]

Listo. Modelos en final/07_resultados_finales.csv: ['gemma2_2b', 'llama_3b', 'llama_8b', 'mistral_7b', 'phi35_mini', 'phi3_mini', 'qwen_1_5b', 'qwen_3b', 'qwen_7b']


,modelo,accuracy,f1_macro,pct_predicciones_invalidas
0,qwen_1_5b,0.7892,0.745879,4.93
1,llama_3b,0.7932,0.738628,5.41
2,qwen_3b,0.7987,0.750618,4.93
3,phi35_mini,0.6872,0.614846,7.98
4,phi3_mini,0.6712,0.590083,7.97
5,mistral_7b,0.7515,0.736744,10.63
6,llama_8b,0.7987,0.752575,5.42
7,qwen_7b,0.7129,0.742600,17.01
8,gemma2_2b,0.8248,0.776153,1.69
